# Principio de Responsabilidad Única (SRP)

Una clase debe tener una sola razón para cambiar. Se contrasta una clase que mezcla responsabilidades con un diseño separado.

## Sin aplicar SRP

`GestorPedido` mezcla cálculo, persistencia y notificación. Un cambio en precios, almacenamiento o mensajes obliga a editar la misma clase.

In [ ]:
class GestorPedido:
    def __init__(self, cliente, productos):
        self.cliente = cliente
        self.productos = productos
        self.estado = 'nuevo'
        self.registros = []

    def calcular_total(self):
        return sum(precio * cantidad for _, precio, cantidad in self.productos)

    def guardar(self):
        registro = {'cliente': self.cliente, 'total': self.calcular_total()}
        self.registros.append(registro)
        return registro

    def enviar_confirmacion(self):
        self.estado = 'confirmado'
        return f'Correo a {self.cliente}: total ${self.calcular_total():,.0f}'

pedido_mal = GestorPedido('ana@email.com', [('Hamburguesa', 18000, 2), ('Jugo', 6000, 1)])
print(pedido_mal.guardar())
print(pedido_mal.enviar_confirmacion())

## Aplicando SRP

`Pedido` representa datos, `CalculadoraPedido` calcula, `RepositorioPedidos` almacena, `NotificadorPedido` comunica y `ServicioPedido` únicamente coordina.

In [ ]:
class Pedido:
    def __init__(self, cliente, productos):
        self.cliente = cliente
        self.productos = productos
        self.estado = 'nuevo'
    def cambiar_estado(self, estado):
        self.estado = estado; return self.estado
    def cantidad_productos(self):
        return sum(cantidad for _, _, cantidad in self.productos)

class CalculadoraPedido:
    def __init__(self, impuesto=0.0, descuento=0.0):
        self.impuesto = impuesto; self.descuento = descuento
    def subtotal(self, pedido):
        return sum(precio * cantidad for _, precio, cantidad in pedido.productos)
    def total(self, pedido):
        return self.subtotal(pedido) * (1 + self.impuesto) * (1 - self.descuento)

class RepositorioPedidos:
    def __init__(self, tabla='pedidos', registros=None):
        self.tabla = tabla; self.registros = registros if registros is not None else []
    def guardar(self, pedido, total):
        self.registros.append({'cliente': pedido.cliente, 'total': total}); return self.registros[-1]
    def listar(self):
        return list(self.registros)

class NotificadorPedido:
    def __init__(self, remitente, canal='correo'):
        self.remitente = remitente; self.canal = canal
    def construir(self, pedido, total):
        return f'Hola {pedido.cliente}, pedido confirmado por ${total:,.0f}'
    def enviar(self, pedido, total):
        return f'[{self.canal}] {self.remitente}: {self.construir(pedido, total)}'

class ServicioPedido:
    def __init__(self, calculadora, repositorio, notificador):
        self.calculadora = calculadora; self.repositorio = repositorio; self.notificador = notificador; self.procesados = 0
    def procesar(self, pedido):
        total = self.calculadora.total(pedido); pedido.cambiar_estado('confirmado')
        self.repositorio.guardar(pedido, total); self.procesados += 1
        return self.notificador.enviar(pedido, total)
    def resumen(self):
        return {'procesados': self.procesados, 'guardados': len(self.repositorio.listar())}

pedido = Pedido('ana@email.com', [('Hamburguesa', 18000, 2), ('Jugo', 6000, 1)])
servicio = ServicioPedido(CalculadoraPedido(0.08), RepositorioPedidos(), NotificadorPedido('pedidos@comida.co'))
print(servicio.procesar(pedido)); print(servicio.resumen())

Cada clase tiene una única razón para cambiar; modificar el almacenamiento no afecta el cálculo ni la notificación.